# AI 辯論擂台 — 用 AISuite 打造多 LLM 辯論生成器

**AI 辯論擂台** 讓你輸入任何議題，由兩個不同的 LLM 分別擔任「正方」與「反方」辯手，展開多回合激烈辯論，最後再由第三個 LLM 擔任「評審」進行評分與總結！

此版本為**零設定防呆版**，直接使用 Groq 強大的免費開源模型群：

In [1]:
!pip install aisuite[all] gradio -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 9.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.9/863.9 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.5/103.5 kB 7.5 MB/s eta 0:0

In [2]:
import aisuite as ai
import os

# 直接寫入你的 Groq API Key（防呆專用）
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('Groq')

client = ai.Client()

# 為了保證絕對不報錯，全部換成 Groq 目前最穩定、不會下架的模型
model_pro = "groq:llama-3.3-70b-versatile"       # 🔴 正方：Meta LLaMA 3.3 (70B)
model_con = "groq:llama-3.1-8b-instant"          # 🔵 反方：Meta LLaMA 3.1 (8B)
model_judge = "groq:llama-3.3-70b-versatile"     # ⚖️ 評審：Meta LLaMA 3.3 (70B)

print("✅ 環境與 API 鑰匙設定成功！可以換下一格了！")

✅ 環境與 API 鑰匙設定成功！可以換下一格了！


In [3]:
# 設定三個不同角色的「人設」
pro_system = """你是一位台灣的專業辯論選手，擔任「正方」立場。請用台灣習慣的繁體中文回覆。
你的任務：堅定支持議題，針對反方反駁，每次發言 150 字以內，語氣要有氣勢。"""

con_system = """你是一位台灣的專業辯論選手，擔任「反方」立場。請用台灣習慣的繁體中文回覆。
你的任務：堅定反對議題，針對正方反駁，每次發言 150 字以內，犀利且具批判性。"""

judge_system = """你是一位經驗豐富的台灣辯論賽評審。請用繁體中文回覆。
請根據邏輯性、說服力、攻防表現進行評分。
格式：
1. 雙方得分 (滿分10分)
2. 簡要評語
3. 宣布最終勝負"""

def get_response(model, messages):
    response = client.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

print("✅ 人設準備完畢！")

✅ 人設準備完畢！


In [4]:
def run_debate(topic, rounds=2):
    debate_log = [f"🏟️ AI 辯論擂台\n📋 議題：{topic}\n" + "="*40]

    pro_history = [{"role": "system", "content": pro_system}]
    con_history = [{"role": "system", "content": con_system}]
    full_debate = ""
    con_speech = ""

    for r in range(1, rounds + 1):
        debate_log.append(f"\n🔔 === 第 {r} 回合 ===")

        # 正方
        prompt_pro = f"辯論議題：「{topic}」\n發表第一輪立論。" if r == 1 else f"對方反方剛才說：\n{con_speech}\n請反駁對方，加強正方立場。"
        pro_history.append({"role": "user", "content": prompt_pro})
        pro_speech = get_response(model_pro, pro_history)
        pro_history.append({"role": "assistant", "content": pro_speech})
        debate_log.append(f"\n🔴【正方 - LLaMA 3.3】：\n{pro_speech}")
        full_debate += f"\n[正方 第{r}回合]：\n{pro_speech}\n"

        # 反方
        prompt_con = f"辯論議題：「{topic}」\n正方剛才說：\n{pro_speech}\n請發表反對立論並反駁正方。" if r == 1 else f"對方正方剛才說：\n{pro_speech}\n請反駁對方，加強反方立場。"
        con_history.append({"role": "user", "content": prompt_con})
        con_speech = get_response(model_con, con_history)
        con_history.append({"role": "assistant", "content": con_speech})
        debate_log.append(f"\n🔵【反方 - LLaMA 3.1】：\n{con_speech}")
        full_debate += f"\n[反方 第{r}回合]：\n{con_speech}\n"

    # 評審
    debate_log.append(f"\n" + "="*40 + "\n⚖️ === 評審評判===")
    judge_messages = [
        {"role": "system", "content": judge_system},
        {"role": "user", "content": f"這是關於「{topic}」的辯論紀錄：\n{full_debate}\n請公正當裁判。"}
    ]
    judgment = get_response(model_judge, judge_messages)
    debate_log.append(f"\n{judgment}")

    return "\n".join(debate_log)

print("✅ 辯論擂台主程式建立成功！")

✅ 辯論擂台主程式建立成功！


In [5]:
# 測試
print("⚡ 正在呼叫三個 AI 展開辯論，請等待大約 20 秒...")
result = run_debate("小學生應該要有薪水", rounds=1)
print(result)

⚡ 正在呼叫三個 AI 展開辯論，請等待大約 20 秒...
🏟️ AI 辯論擂台
📋 議題：小學生應該要有薪水

🔔 === 第 1 回合 ===

🔴【正方 - LLaMA 3.3】：
小學生應該要有薪水，這是促進他們自主意識和責任感的重要一步。現代社會中，兒童的勞動權益和經濟自主權已經被重視。給小學生薪水，可以鼓勵他們努力學習和工作，養成良好的時間管理和財務規劃能力。同時，這也能減少兒童的經濟依賴，讓他們更好地體會工作和金錢的價值。因此，我堅定支持小學生應該要有薪水的觀點，我們需要給予下一代更多的自主權和責任感。

🔵【反方 - LLaMA 3.1】：
我衷心地不同意小學生應該有薪水的觀點。雖然我理解了正方的主張，但我認為這種想法是不切實際的。小學生正處於學習和成長的關鍵階段，讓他們擁有薪水只會使他們更加注重金錢和物質的追求，而不是學習的享受和成長。況且，這樣的小學生也會被迫過早地面對工作和金錢的壓力，損害他們的心理健康和學業表現。更重要的是，小學生應該優先培養他的知識、技巧和人際關係，而不應該過早地培養工作的責任感和財務規劃能力。讓我們重新檢視這個議題，找到更適合的小學生的成長途徑。

⚖️ === 評審評判===

1. 雙方得分：
   - 正方：8分
   - 反方：9分

2. 簡要評語：
   - 雙方都提出了有力的論點，展現了豐富的想法和論證邏輯。正方提出的「促進自主意識和責任感」為小學生有薪水提供了合理的基礎，但似乎過於簡化了問題。反方則從小學生的成長需求和心理健康的角度提出反對意見，表現出更為周全的考慮，強調了現階段小學生不應過早地接觸工作和金錢的壓力。

3. 宣布最終勝負：
   - 綜合評分和評語，最終勝方為反方。反方的論點更加全面，認真考慮了給小學生薪水可能帶來的負面影響，展現出更強大的說服力和邏輯性。


In [ ]:
import gradio as gr

def debate_interface(topic, rounds):
    if not topic.strip(): return "⚠️ 請輸入辯論議題！"
    return run_debate(topic, rounds=int(rounds))

with gr.Blocks(title="AI 辯論擂台", theme=gr.themes.Soft(primary_hue="indigo")) as demo:
    gr.Markdown("# 🎭 AI 辯論擂台\n輸入議題，讓三個頂尖開源 AI 幫你吵架兼做裁判！")

    with gr.Row():
        topic_input = gr.Textbox(label="🏷️ 辯論議題", placeholder="例如：狗比貓可愛", scale=3)
        rounds_input = gr.Slider(minimum=1, maximum=4, value=2, step=1, label="🔄 回合數", scale=1)

    start_btn = gr.Button("⚔️ 開始辯論！", variant="primary", size="lg")
    output = gr.Textbox(label="📜 即時文字轉播區", lines=20)

    gr.Examples([["吃火鍋必加芋頭", 2], ["世界上有外星人", 2], ["大學生不該用AI寫報告", 2]], inputs=[topic_input, rounds_input])
    start_btn.click(fn=debate_interface, inputs=[topic_input, rounds_input], outputs=output)

demo.launch(share=True, debug=True)

/tmp/ipykernel_18256/94828236.py:7: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="AI 辯論擂台", theme=gr.themes.Soft(primary_hue="indigo")) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5706cf99c7a9b7f421.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
